# MEMORY

**STANDALONE MEMORY USING EXTRACT_MEMORIES**

In [7]:
from crewai import Memory

memory = Memory()

text = input("Enter your meeting notes: ")

# Extract important facts
facts = memory.extract_memories(text)

# Store extracted facts
for fact in facts:
    memory.remember(fact)

# Display stored facts
print("\nExtracted Memories:")

for fact in facts:
    print("-", fact)

Enter your meeting notes:  The team decided to use Python for the project. The deadline is December 15. Monisha will handle the database.



Extracted Memories:
- The team decided to use Python for the project.
- The deadline is December 15.
- Monisha will handle the database.


**MEMORY WITH A CREW**

In [10]:
from crewai import Agent, Task, Crew

agent = Agent(
    role="Teacher",
    goal="Provide student information",
    backstory="A helpful teacher"
)

task = Task(
    description="Tell me the name and role of the student: Vidhya, AI Developer.",
    expected_output="Student name and role",
    agent=agent
)

crew = Crew(
    agents=[agent],
    tasks=[task],
    memory=True
)

result = await crew.kickoff_async()
print("Crew memory enabled:", crew.memory)

print("===== RESULT =====")
print(result)

Crew memory enabled: True
===== RESULT =====
The student name is Vidhya, and her role is AI Developer.


**MEMORY WITH AN AGENT**

In [12]:
from crewai import Agent, Task, Crew

# Create an agent
agent = Agent(
    role="Student Assistant",
    goal="Remember and provide student information",
    backstory="A helpful assistant that remembers student details.",
    memory=True
)

# Task
task = Task(
    description="Remember that Vidhya is an AI Developer learning CrewAI.",
    expected_output="Confirm the student information.",
    agent=agent
)

# Create Crew
crew = Crew(
    agents=[agent],
    tasks=[task]
)

# Run the task
result = await crew.kickoff_async()

# Print result
print("===== RESULT =====")
print(result)

===== RESULT =====
The student information is as follows:

- The student's name is Vidhya.
- Vidhya's role is AI Developer.
- Vidhya is learning and working on the Crew AI project.


**MEMORY INSIDE A FLOW**

In [14]:
from crewai.flow.flow import Flow, listen, start

class StudentFlow(Flow):

    @start()
    def store_data(self):
        student_info = "The student's name is Vidhya."    
        self.remember(student_info, scope="/students")     # Store information in Flow memory
        return student_info

    @listen(store_data)
    def retrieve_data(self, student_info):
        past = self.recall("student name")         # Recall stored information
        context = "\n".join(f"- {m.record.content}" for m in past)
        return f"""
Student Information:
New data: {student_info}

Retrieved Memory:
{context}
"""
flow = StudentFlow()
flow.plot()
result = flow.kickoff()

print("\n===== FINAL RESULT =====")
print(result)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: StudentFlow                                                                                              │
│  ID: a7c94dc8-c33c-4698-b6e2-06d65d4121cb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: StudentFlow                                                                                              │
│  ID: a7c94dc8-c33c-4698-b6e2-06d65d4121cb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: store_data                                                                                             │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: store_data                                                                                             │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: retrieve_data                                                                                          │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: retrieve_data                                                                                          │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: StudentFlow                                                                                              │
│  ID: a7c94dc8-c33c-4698-b6e2-06d65d4121cb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== FINAL RESULT =====

Student Information:
New data: The student's name is Vidhya.

Retrieved Memory:
- The student's name is Vidhya.



**MEMORY SCOPE - A restricted/separate section of memory used to organize and isolate information.**

In [17]:
from crewai.memory import Memory

# Create memory
memory = Memory()

# Store memories in different scopes
memory.remember("The student's name is Vidhya.", scope="/STUDENT")
memory.remember("The student's age is 25.", scope="/STUDENT")

memory.remember("The teacher's name is Monisha.", scope="/TEACHER")
memory.remember("The teacher teaches Python.", scope="/TEACHER")

memory.remember("The course name is Python.", scope="/COURSE")
memory.remember("The course duration is 6 months.", scope="/COURSE")

memory.remember("The project name is CrewAI.", scope="/PROJECT")
memory.remember("The project uses AI agents.", scope="/PROJECT")


# Function to retrieve memories from a scope
def show_memory(scope, query):
    results = memory.recall(query, scope=scope)

    print(f"\n{scope} Memory:")

    for match in results:
        print(match.record.content)


# Retrieve from multiple scopes
show_memory("/STUDENT", "student")
show_memory("/TEACHER", "teacher")
show_memory("/COURSE", "course")
show_memory("/PROJECT", "project")


/STUDENT Memory:
The student's name is Vidhya.
The student's age is 25.

/TEACHER Memory:
The teacher's name is Monisha.
The teacher teaches Python.

/COURSE Memory:
The course is Python.
The course name is Python.
The course duration is 6 months.

/PROJECT Memory:
The project name is CrewAI.
The project uses AI agents.


**HIERARCHICAL SCOPE**

<div align="center">
    <img src="images/Hierarchical Scope.png" width="500" height="2000">
</div>

In [24]:
from crewai import Memory

memory = Memory()

# Store memories
memory.remember(scope="PROJECT/AI", content="developer → Vidhya")
memory.remember(scope="PROJECT/AI", content="language → Python")
memory.remember(scope="PROJECT/WEB", content="developer → Monisha")
memory.remember(scope="PROJECT/WEB", content="framework → Flask")

# Retrieve AI memories
ai = memory.recall("developer", scope="PROJECT/AI")

# Retrieve WEB memories
web = memory.recall("developer", scope="PROJECT/WEB")

print("AI:")
for m in ai:
    print(" ", m.record.content)
print("\nWEB:")
for m in web:
    print(" ", m.record.content)

AI:
  developer → Vidhya
  language → Python

WEB:
  developer → Monisha
  framework → Flask


**A MEMORY TREE in CrewAI shows how stored memories are organized into scopes and subscopes, similar to folders and subfolders.**

<div align="center">
    <img src="images/memorytree.png" width="500" height="2000">
</div>

In [9]:
from tempfile import mkdtemp
from crewai import Memory

# Create a new, separate memory store
memory = Memory(storage=mkdtemp(prefix="project_memory_"))
memory.remember(scope="/PROJECT/AI", content="developer → Vidhya")
memory.remember(scope="/PROJECT/AI", content="language → Python")
memory.remember(scope="/PROJECT/WEB", content="developer → Monisha")
memory.remember(scope="/PROJECT/WEB", content="framework → Flask")
print(memory.tree())

/ (4 records)
  /PROJECT (4 records)
    /PROJECT/AI (2 records)
    /PROJECT/WEB (2 records)


**MEMORY SLICES - It is a selected view of one or more memory scopes. It allows an agent to access memories from several different branches without accessing the entire memory.**

**READ-ONLY SLICES**

<div align="center">
    <img src="images/Read-only.png" width="700" height="2000">
</div>

In [12]:
from crewai import Memory

# Create memory
memory = Memory()

# Store AI project memories
memory.remember(scope="PROJECT/AI", content="developer → Vidhya")
memory.remember(scope="PROJECT/AI", content="language → Python")

# Store WEB project memory
memory.remember(scope="PROJECT/WEB", content="developer → Monisha")

# Create a read-only slice containing only PROJECT/AI
ai_slice = memory.slice(scopes=["PROJECT/AI"], read_only=True)

# Retrieve memories only from PROJECT/AI
matches = ai_slice.recall("developer and programming language", limit=10, depth="shallow")

# Display the slice configuration
print("AI Memory Slice - Read Only:")
print(ai_slice)

# Display the retrieved memories
print("\nRetrieved AI Memories:")

if matches:
    for match in matches:
        print(" ", match.record.content)
else:
    print("  No matching memories found.")

AI Memory Slice - Read Only:
memory_kind='slice' scopes=['PROJECT/AI'] categories=None read_only=True

Retrieved AI Memories:
  developer → Vidhya
  language → Python


**READ-WRITE SLICES**

<div align="center">
    <img src="images/Read-write.png" width="700" height="2000">
</div>

In [13]:
from crewai import Memory

# Create memory
memory = Memory()

# Store initial memories
memory.remember(scope="PROJECT/AI", content="developer → Vidhya")
memory.remember(scope="PROJECT/WEB", content="developer → Monisha")

# Create a READ-WRITE slice
project_slice = memory.slice(scopes=["PROJECT/AI", "PROJECT/WEB"], read_only=False)

# Add new memories through the slice
project_slice.remember(scope="PROJECT/AI", content="language → Python")    # Target Scope
project_slice.remember(scope="PROJECT/WEB", content="framework → Flask")   # Target Scope

# Retrieve memories from both selected scopes
matches = project_slice.recall("developer, programming language and web framework", limit=10, depth="shallow")

# Display slice information
print("Project Memory Slice - Read and Write:")
print(project_slice)

# Display retrieved memories
print("\nRetrieved Project Memories:")

if matches:
    for match in matches:
        print(" ", match.record.content)
else:
    print("  No matching memories found.")

Project Memory Slice - Read and Write:
memory_kind='slice' scopes=['PROJECT/AI', 'PROJECT/WEB'] categories=None read_only=False

Retrieved Project Memories:
  developer → Monisha
  developer → Vidhya
  framework → Flask
  language → Python
